<a href="https://colab.research.google.com/github/Singh543214/ai-mentor-portfolio/blob/main/Day2_ResumeExtractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-genai pydantic
import os, getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

Gemini API key: ··········


In [2]:
from pydantic import BaseModel
from typing import List, Optional

class Education(BaseModel):
    degree: str
    institution: str
    year: int

class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    education: List[Education]
    skills: List[str]
    projects: List[str] = []
    experience_years: float

In [3]:
from google import genai
from pydantic import ValidationError

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

def extract_resume(raw_text: str, max_retries: int = 1) -> Resume:
    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f'Extract a Resume JSON from this text. Return ONLY JSON, no markdown.\n\n{raw_text}',
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)
        except ValidationError as e:
            if attempt == max_retries:
                raise
            # Retry once with the broken JSON in the prompt
            fix_prompt = (f'Fix this JSON to match schema. Errors: {e}. '
                          f'Original: {resp.text}')
            resp = client.models.generate_content(
                model='gemini-2.5-flash', contents=fix_prompt,
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)

In [19]:
# Load sample résumés from the lab kit
with open('data/sample_resumes.txt') as f:
    resumes = [r.strip() for r in f.read().split('---') if r.strip()]

print(f'Loaded {len(resumes)} sample résumés')

results = []
for i, r in enumerate(resumes[:3]):
    try:
        parsed = extract_resume(r)
        results.append(parsed)
        print(f'\nRésumé {i+1}: {parsed.name} — {len(parsed.skills)} skills, '
              f'{parsed.experience_years} years exp')
    except Exception as e:
        print(f'\nRésumé {i+1}: FAILED — {type(e).__name__}: {str(e)[:200]}')

# Print full first result
if results:
    print('\n=== Full first result ===')
    print(results[0].model_dump_json(indent=2))

Loaded 6 sample résumés

Résumé 1: Candidate A — 9 skills, 6.0 years exp

Résumé 2: John Doe — 3 skills, 6.0 years exp

Résumé 3: Candidate C — 9 skills, 5.0 years exp

=== Full first result ===
{
  "name": "Candidate A",
  "email": "candidate.a@email.com",
  "phone": "+1-555-010-1001",
  "education": [
    {
      "degree": "B.Sc. Computer Science",
      "institution": "State University",
      "year": 2018
    }
  ],
  "skills": [
    "Python",
    "JavaScript",
    "React",
    "Node.js",
    "PostgreSQL",
    "AWS",
    "Docker",
    "Git",
    "Agile/Scrum"
  ],
  "projects": [
    "Built a real-time chat application using WebSockets and React",
    "Developed a REST API for an inventory management system using Django",
    "Created a CI/CD pipeline automation tool using GitHub Actions and Docker"
  ],
  "experience_years": 6.0
}


In [20]:
# Empty string — should fail gracefully, not crash
try:
    bad = extract_resume('')
    print('Unexpected success:', bad.model_dump_json())
except Exception as e:
    print('Caught gracefully:', type(e).__name__)
    print('Message:', str(e)[:200])

Caught gracefully: ServerError
Message: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
